In [3]:
bamfile = "/mnt/c/Users/julien.mortier/Downloads/possorted_genome_bam.bam"

total_umis = 0
unique_umis = set()

with pysam.AlignmentFile(bamfile, "rb") as bam:
    for read in bam:
        if read.has_tag("UB") and read.has_tag("GN"):
            umi = read.get_tag("UB")
            gene = read.get_tag("GN")
            cell = read.get_tag("CB") if read.has_tag("CB") else "NA"
            
            total_umis += 1
            unique_umis.add((cell, umi, gene))

print("Total UMIs:", total_umis)
print("Unique UMIs:", len(unique_umis))
print("Sequencing saturation:", 1 - len(unique_umis)/total_umis)


Total UMIs: 74304550
Unique UMIs: 33137962
Sequencing saturation: 0.5540251303587734


In [7]:
import pysam
import pandas as pd
import random

bamfile = "/mnt/c/Users/julien.mortier/Downloads/possorted_genome_bam.bam"

# Define subsampling levels (number of reads)
subsample_levels = [10_000_000, 20_000_000, 30_000_000]

# Step 1: Count total reads
print("Counting total reads...")
with pysam.AlignmentFile(bamfile, "rb") as bam:
    total_reads = sum(1 for _ in bam)
print(f"Total reads in BAM: {total_reads:,}")

# Step 2: Generate random indices for each subsample
subsample_indices = {}
for n in subsample_levels:
    if n > total_reads:
        n = total_reads
    subsample_indices[n] = set(random.sample(range(total_reads), n))

# Step 3: Process BAM and compute metrics
results = []

with pysam.AlignmentFile(bamfile, "rb") as bam:
    for n in subsample_levels:
        total_umis = 0
        unique_umis = set()
        for i, read in enumerate(bam):
            if i in subsample_indices[n]:
                if read.has_tag("UB") and read.has_tag("GN"):
                    umi = read.get_tag("UB")
                    gene = read.get_tag("GN")
                    cell = read.get_tag("CB") if read.has_tag("CB") else "NA"
                    total_umis += 1
                    unique_umis.add((cell, umi, gene))
        saturation = 1 - len(unique_umis)/total_umis if total_umis > 0 else 0
        results.append({
            "subsample_reads": n,
            "total_umis": total_umis,
            "unique_umis": len(unique_umis),
            "sequencing_saturation": saturation
        })

df = pd.DataFrame(results)
df.to_csv("bam_subsample_metrics.csv", index=False)
print("Metrics saved to bam_subsample_metrics.csv")


Counting total reads...


[W::hts_idx_load3] The index file is older than the data file: /mnt/c/Users/julien.mortier/Downloads/possorted_genome_bam.bam.bai


Total reads in BAM: 74,634,733


[W::hts_idx_load3] The index file is older than the data file: /mnt/c/Users/julien.mortier/Downloads/possorted_genome_bam.bam.bai


Metrics saved to bam_subsample_metrics.csv
